# Verify the scientific runtime

Probe the configured runtime in a separate process. Checks all 119 package versions, pip consistency and CPU arithmetic. No Unity launch, model loading or training. Missing configuration is saved as BLOCKED; a configured runtime mismatch raises an error.


In [1]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)

print('Workspace ready:', ROOT)


Configuration helpers loaded. No runtime probe or installation has been executed.
Workspace ready: /home/obidit/900_PrePreDefense/trace-lab


## Inspect the configured interpreter


In [2]:
assets = load_assets(ROOT)
print(json.dumps(asset_status(assets), indent=2))


[
  {
    "asset": "runtime_python",
    "kind": "executable",
    "path": "/home/obidit/miniconda3/envs/affect-envs/bin/python3.9",
    "status": "AVAILABLE"
  },
  {
    "asset": "conda_executable",
    "kind": "executable",
    "path": "/home/obidit/miniconda3/bin/conda",
    "status": "AVAILABLE"
  },
  {
    "asset": "runtime_archives",
    "kind": "directory",
    "path": "/home/obidit/900_PrePreDefense/phase1/environment",
    "status": "AVAILABLE"
  },
  {
    "asset": "upstream_source",
    "kind": "directory",
    "path": "/home/obidit/900_PrePreDefense/trace-lab/assets/upstream_source",
    "status": "AVAILABLE"
  },
  {
    "asset": "unity_player",
    "kind": "executable",
    "path": "/home/obidit/900_PrePreDefense/trace-lab/assets/unity/solid.x86_64",
    "status": "MISSING"
  },
  {
    "asset": "model_directory",
    "kind": "directory",
    "path": "/home/obidit/900_PrePreDefense/trace-lab/assets/models",
    "status": "MISSING"
  },
  {
    "asset": "dataset_director

## Verify and save a uniquely named local record


In [3]:
state = next(row for row in asset_status(assets) if row['asset'] == 'runtime_python')
if state['status'] != 'AVAILABLE':
    result = {'status': 'BLOCKED', 'reason': state, 'scope': 'Configure runtime_python; no runtime check executed.'}
else:
    probe = runtime_probe(require_asset(assets, 'runtime_python'))
    result = compare_runtime(ROOT, probe)
    result['probe'] = probe
record = save_record(ROOT, 'runtime_verification', result)
print(json.dumps({key: value for key, value in result.items() if key != 'probe'}, indent=2))
print('Local evidence:', record)
if result['status'] == 'FAIL':
    raise AssertionError('Runtime differs from the preserved environment; see saved record')


{
  "status": "PASS",
  "checks": {
    "python_version": true,
    "package_versions": true,
    "pip_check": true,
    "cpu_arithmetic": true,
    "cpu_device": true,
    "retained_torch": true
  },
  "expected_distributions": 119,
  "missing": [],
  "extra": [],
  "changed": {}
}
Local evidence: /home/obidit/900_PrePreDefense/trace-lab/outputs/setup/20260913T134948Z-839d1c86/runtime_verification.json
